<figure>
<center>
<img src='https://www.economicas.uba.ar/wp-content/uploads/2020/08/cropped-logo_FCE.png' />
</center>
</figure>

# **Universidad de Buenos Aires**
## **Facultad de Ciencias Económicas**

### **Taller de Programación para el Análisis de Datos**

# 🛒 Estadística Descriptiva Aplicada
## Dataset: Online Retail UK — UCI Machine Learning Repository

---

### Contexto de negocio

Trabajamos con datos reales de una **tienda mayorista online del Reino Unido** (2010–2011).
La empresa vende artículos de regalo a minoristas de todo el mundo.

| Variable | Descripción |
|---|---|
| `InvoiceNo` | Número de factura (si empieza con 'C' → devolución) |
| `StockCode` | Código de producto |
| `Description` | Nombre del producto |
| `Quantity` | Unidades por transacción (negativo = devolución) |
| `InvoiceDate` | Fecha y hora de la transacción |
| `UnitPrice` | Precio unitario en libras esterlinas (£) |
| `CustomerID` | Identificador del cliente |
| `Country` | País del cliente |

**Fuente:** [UCI Machine Learning Repository — Online Retail](https://archive.ics.uci.edu/dataset/352/online+retail)

---
### Objetivos de la clase
- Aplicar medidas de posición, dispersión y asimetría sobre datos reales
- Ver por qué la **media no es suficiente** en distribuciones asimétricas
- Practicar limpieza de datos antes del análisis
- Comparar distribuciones entre grupos (países, meses)

---
## 0. Carga del dataset

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# El dataset está disponible en el repositorio UCI.
# Opción A: carga directa desde URL (requiere conexión a internet)
# Opción B: descargá el archivo desde https://archive.ics.uci.edu/dataset/352/online+retail
#           y descomprimilo en la misma carpeta que esta notebook.

URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"

try:
    df_raw = pd.read_excel(URL, engine='openpyxl')
    print("Dataset cargado desde URL")
except Exception:
    try:
        df_raw = pd.read_excel("Online Retail.xlsx", engine='openpyxl')
        print("Dataset cargado desde archivo local")
    except FileNotFoundError:
        print("No se pudo cargar el dataset.")
        print("Descargalo desde: https://archive.ics.uci.edu/dataset/352/online+retail")
        print("y colocá 'Online Retail.xlsx' en la misma carpeta que esta notebook.")
        df_raw = None

if df_raw is not None:
    print(f"\nFilas: {len(df_raw):,}")
    print(f"Columnas: {df_raw.columns.tolist()}")
    df_raw.head()

C:\Users\cmarc\AppData\Local\Temp\ipykernel_20396\295543553.py:3: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.2)
  from scipy import stats


✅ Dataset cargado desde URL

Filas: 541,909
Columnas: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


---
## 1. Exploración inicial y limpieza

Los datos reales casi siempre tienen problemas. Antes de calcular cualquier estadística, entendemos qué tenemos.

In [2]:
print("── Tipos de datos ──")
print(df_raw.dtypes)
print(f"\n── Valores nulos por columna ──")
nulos = df_raw.isnull().sum()
print(nulos[nulos > 0])
print(f"\nTotal de filas originales: {len(df_raw):,}")

── Tipos de datos ──
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

── Valores nulos por columna ──
Description      1454
CustomerID     135080
dtype: int64

Total de filas originales: 541,909


In [3]:
# ── Decisiones de limpieza ────────────────────────────────────────
#
# 1. Eliminar filas sin CustomerID (no identificables)
# 2. Separar devoluciones (InvoiceNo que empieza con 'C') para análisis aparte
# 3. Eliminar precios = 0 (productos gratuitos / errores de carga)
# 4. Eliminar cantidades <= 0 en el dataset de ventas

# Separamos devoluciones ANTES de limpiar
devoluciones = df_raw[df_raw['InvoiceNo'].astype(str).str.startswith('C')].copy()
ventas_raw   = df_raw[~df_raw['InvoiceNo'].astype(str).str.startswith('C')].copy()

# Aplicamos limpieza sobre ventas
df = ventas_raw.copy()
df = df.dropna(subset=['CustomerID'])
df = df[df['UnitPrice'] > 0]
df = df[df['Quantity'] > 0]

# Feature engineering: Revenue = ingresos por línea de factura
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# Extraemos componentes de fecha
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Mes']         = df['InvoiceDate'].dt.month
df['DiaSemana']   = df['InvoiceDate'].dt.day_name()

print(f"Filas originales : {len(df_raw):,}")
print(f"Devoluciones     : {len(devoluciones):,}")
print(f"Ventas limpias   : {len(df):,}")
print(f"\nPeriodo: {df['InvoiceDate'].min().date()}  →  {df['InvoiceDate'].max().date()}")
print(f"Países  : {df['Country'].nunique()}")
print(f"Clientes: {df['CustomerID'].nunique():,}")
print(f"Productos: {df['StockCode'].nunique():,}")
df.head()

Filas originales : 541,909
Devoluciones     : 9,288
Ventas limpias   : 397,884

Periodo: 2010-12-01  →  2011-12-09
Países  : 37
Clientes: 4,338
Productos: 3,665


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,Mes,DiaSemana
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,12,Wednesday
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,12,Wednesday
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,12,Wednesday
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,12,Wednesday
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,12,Wednesday


---
## 2. Análisis descriptivo — Precio unitario (`UnitPrice`)

Empezamos por el precio. En datos de ventas, la distribución de precios suele ser **muy asimétrica**: la mayoría de los productos son baratos, pero algunos tienen precios muy altos.

In [4]:
precios = df['UnitPrice']

print("═" * 50)
print("  PRECIO UNITARIO (£)")
print("─" * 50)
print(f"  n                  : {len(precios):,}")
print(f"── Posición ──")
print(f"  Media              : £{precios.mean():.4f}")
print(f"  Mediana            : £{precios.median():.4f}")
print(f"  Moda               : £{precios.mode()[0]:.4f}")
print(f"  P10 / P25 / P75 / P90: £{precios.quantile(0.10):.2f} / £{precios.quantile(0.25):.2f} / £{precios.quantile(0.75):.2f} / £{precios.quantile(0.90):.2f}")
print(f"── Dispersión ──")
print(f"  Mín / Máx          : £{precios.min():.2f} / £{precios.max():.2f}")
print(f"  Rango              : £{precios.max() - precios.min():.2f}")
print(f"  RIQ                : £{precios.quantile(0.75) - precios.quantile(0.25):.4f}")
print(f"  Desvío estándar    : £{precios.std():.4f}")
print(f"  CV                 : {precios.std()/precios.mean()*100:.1f}%")
print(f"── Asimetría ──")
print(f"  Fisher G1          : {precios.skew():.4f}")
print("═" * 50)

print("\n💡 Observación:")
print(f"   La media (£{precios.mean():.2f}) es MUY superior a la mediana (£{precios.median():.2f}).")
print(f"   Esto indica que unos pocos productos de precio muy alto 'tiran' la media hacia arriba.")
print(f"   CV = {precios.std()/precios.mean()*100:.0f}% → dispersión altísima respecto a la media.")

══════════════════════════════════════════════════
  PRECIO UNITARIO (£)
──────────────────────────────────────────────────
  n                  : 397,884
── Posición ──
  Media              : £3.1165
  Mediana            : £1.9500
  Moda               : £1.2500
  P10 / P25 / P75 / P90: £0.55 / £1.25 / £3.75 / £6.35
── Dispersión ──
  Mín / Máx          : £0.00 / £8142.75
  Rango              : £8142.75
  RIQ                : £2.5000
  Desvío estándar    : £22.0979
  CV                 : 709.1%
── Asimetría ──
  Fisher G1          : 204.0327
══════════════════════════════════════════════════

💡 Observación:
   La media (£3.12) es MUY superior a la mediana (£1.95).
   Esto indica que unos pocos productos de precio muy alto 'tiran' la media hacia arriba.
   CV = 709% → dispersión altísima respecto a la media.


In [5]:
# ¿Cuánto representa el P90 vs el máximo?
p90 = precios.quantile(0.90)
p99 = precios.quantile(0.99)

print(f"El 90% de los productos tiene precio ≤ £{p90:.2f}")
print(f"El 99% de los productos tiene precio ≤ £{p99:.2f}")
print(f"El precio máximo es £{precios.max():.2f}")
print(f"\nProductos con precio > £{p99:.0f} (top 1%):")
print(df[df['UnitPrice'] > p99][['Description', 'UnitPrice', 'Country']]
      .sort_values('UnitPrice', ascending=False)
      .drop_duplicates('Description')
      .head(10)
      .to_string(index=False))

El 90% de los productos tiene precio ≤ £6.35
El 99% de los productos tiene precio ≤ £14.95
El precio máximo es £8142.75

Productos con precio > £15 (top 1%):
                       Description  UnitPrice        Country
                           POSTAGE    8142.75 United Kingdom
                            Manual    4161.06         France
                    DOTCOM POSTAGE    1599.26 United Kingdom
    PICNIC BASKET WICKER 60 PIECES     649.50 United Kingdom
       VINTAGE RED KITCHEN CABINET     295.00 United Kingdom
      VINTAGE BLUE KITCHEN CABINET     295.00 United Kingdom
     LOVE SEAT ANTIQUE WHITE METAL     195.00 United Kingdom
      REGENCY MIRROR WITH SHUTTERS     165.00 United Kingdom
RUSTIC  SEVENTEEN DRAWER SIDEBOARD     165.00           EIRE
                          CARRIAGE     150.00         France


### ¿Qué pasa si excluimos los outliers?

Comparamos las estadísticas con y sin el top 1% de precios.

In [6]:
precios_sin_outliers = precios[precios <= p99]

print(f"{'Medida':<22} {'Con outliers':>14} {'Sin top 1%':>14} {'Diferencia':>12}")
print("─" * 65)

medidas = [
    ("Media",           precios.mean(),              precios_sin_outliers.mean()),
    ("Mediana",         precios.median(),             precios_sin_outliers.median()),
    ("Desvío estándar", precios.std(),                precios_sin_outliers.std()),
    ("CV (%)",          precios.std()/precios.mean()*100, precios_sin_outliers.std()/precios_sin_outliers.mean()*100),
    ("Asimetría G1",    precios.skew(),               precios_sin_outliers.skew()),
]

for nombre, con, sin in medidas:
    diff_pct = (sin - con) / con * 100
    print(f"{nombre:<22} {con:>14.4f} {sin:>14.4f} {diff_pct:>+11.1f}%")

print("\n💡 La mediana cambia muy poco (-0.X%) → es ROBUSTA ante outliers.")
print("   La media y el desvío se ven mucho más afectados.")

Medida                   Con outliers     Sin top 1%   Diferencia
─────────────────────────────────────────────────────────────────
Media                          3.1165         2.7129       -13.0%
Mediana                        1.9500         1.7900        -8.2%
Desvío estándar               22.0979         2.5355       -88.5%
CV (%)                       709.0635        93.4608       -86.8%
Asimetría G1                 204.0327         1.9346       -99.1%

💡 La mediana cambia muy poco (-0.X%) → es ROBUSTA ante outliers.
   La media y el desvío se ven mucho más afectados.


---
## 3. Análisis descriptivo — Cantidad por transacción (`Quantity`)

In [7]:
cantidad = df['Quantity']

print("═" * 50)
print("  CANTIDAD POR TRANSACCIÓN (unidades)")
print("─" * 50)
print(f"  Media              : {cantidad.mean():.2f} unidades")
print(f"  Mediana            : {cantidad.median():.0f} unidades")
print(f"  Moda               : {cantidad.mode()[0]:.0f} unidades")
print(f"  P25 / P75          : {cantidad.quantile(0.25):.0f} / {cantidad.quantile(0.75):.0f}")
print(f"  RIQ                : {cantidad.quantile(0.75) - cantidad.quantile(0.25):.0f}")
print(f"  Rango              : {cantidad.min()} → {cantidad.max()}")
print(f"  Desvío estándar    : {cantidad.std():.2f}")
print(f"  CV                 : {cantidad.std()/cantidad.mean()*100:.1f}%")
print(f"  Asimetría G1       : {cantidad.skew():.4f}")
print("═" * 50)

# ¿Hay pedidos muy grandes?
p99_q = cantidad.quantile(0.99)
grandes = df[df['Quantity'] >= p99_q][['Description', 'Quantity', 'UnitPrice', 'Country']]
print(f"\nPedidos más grandes (top 1%, Quantity ≥ {p99_q:.0f} unidades):")
print(grandes.sort_values('Quantity', ascending=False).head(8).to_string(index=False))

══════════════════════════════════════════════════
  CANTIDAD POR TRANSACCIÓN (unidades)
──────────────────────────────────────────────────
  Media              : 12.99 unidades
  Mediana            : 6 unidades
  Moda               : 1 unidades
  P25 / P75          : 2 / 12
  RIQ                : 10
  Rango              : 1 → 80995
  Desvío estándar    : 179.33
  CV                 : 1380.7%
  Asimetría G1       : 409.8930
══════════════════════════════════════════════════

Pedidos más grandes (top 1%, Quantity ≥ 120 unidades):
                        Description  Quantity  UnitPrice        Country
        PAPER CRAFT , LITTLE BIRDIE     80995       2.08 United Kingdom
     MEDIUM CERAMIC TOP STORAGE JAR     74215       1.04 United Kingdom
  WORLD WAR 2 GLIDERS ASSTD DESIGNS      4800       0.21 United Kingdom
               SMALL POPCORN HOLDER      4300       0.72 United Kingdom
              EMPIRE DESIGN ROSETTE      3906       0.82 United Kingdom
ESSENTIAL BALM 3.5g TIN IN ENVELO

---
## 4. Análisis descriptivo — Revenue por transacción

El **Revenue** (Quantity × UnitPrice) es la variable de negocio más importante. En e-commerce, la distribución del revenue es casi siempre **extremadamente asimétrica**: pocas transacciones generan la mayor parte de los ingresos.

In [8]:
rev = df['Revenue']

print("═" * 52)
print("  REVENUE POR TRANSACCIÓN (£)")
print("─" * 52)
print(f"  Media              : £{rev.mean():>10.2f}")
print(f"  Mediana            : £{rev.median():>10.2f}")
print(f"  P25 / P75 / P90    : £{rev.quantile(0.25):.2f} / £{rev.quantile(0.75):.2f} / £{rev.quantile(0.90):.2f}")
print(f"  Mín / Máx          : £{rev.min():.2f} / £{rev.max():,.2f}")
print(f"  Desvío estándar    : £{rev.std():>10.2f}")
print(f"  CV                 : {rev.std()/rev.mean()*100:.1f}%")
print(f"  Asimetría G1       : {rev.skew():.4f}")
print("═" * 52)

# Regla de Pareto: ¿el 20% de las transacciones genera el 80% del revenue?
rev_ordenado = rev.sort_values(ascending=False)
rev_total    = rev_ordenado.sum()
n_total      = len(rev_ordenado)

for pct_trans in [0.05, 0.10, 0.20]:
    top_n    = int(n_total * pct_trans)
    top_rev  = rev_ordenado.iloc[:top_n].sum()
    pct_rev  = top_rev / rev_total * 100
    print(f"  Top {pct_trans*100:.0f}% de transacciones → {pct_rev:.1f}% del revenue total")

════════════════════════════════════════════════════
  REVENUE POR TRANSACCIÓN (£)
────────────────────────────────────────────────────
  Media              : £     22.40
  Mediana            : £     11.80
  P25 / P75 / P90    : £4.68 / £19.80 / £35.40
  Mín / Máx          : £0.00 / £168,469.60
  Desvío estándar    : £    309.07
  CV                 : 1380.0%
  Asimetría G1       : 451.4432
════════════════════════════════════════════════════
  Top 5% de transacciones → 43.8% del revenue total
  Top 10% de transacciones → 54.4% del revenue total
  Top 20% de transacciones → 66.5% del revenue total


---
## 5. Análisis por país

Comparamos las estadísticas descriptivas de revenue entre los principales países.

In [9]:
# Top 10 países por revenue total
top_paises = (df.groupby('Country')['Revenue']
                .sum()
                .sort_values(ascending=False)
                .head(10)
                .index.tolist())

df_top = df[df['Country'].isin(top_paises)]

resumen_paises = (df_top.groupby('Country')['Revenue']
                  .agg(
                      transacciones='count',
                      revenue_total='sum',
                      media='mean',
                      mediana='median',
                      desvio='std',
                      asimetria='skew'
                  )
                  .sort_values('revenue_total', ascending=False)
                  .round(2))

resumen_paises['CV (%)'] = (resumen_paises['desvio'] / resumen_paises['media'] * 100).round(1)

print(resumen_paises.to_string())

print("\n💡 Notá que en casi todos los países: media >> mediana → distribución asimétrica positiva.")

                transacciones  revenue_total   media  mediana  desvio  asimetria  CV (%)
Country                                                                                 
United Kingdom         354321     7308391.55   20.63     10.2  326.04     431.79  1580.4
Netherlands              2359      285446.34  121.00     91.8  164.05      13.62   135.6
EIRE                     7236      265545.90   36.70     17.4   83.94      11.15   228.7
Germany                  9040      228867.14   25.32     17.0   35.46       8.44   140.0
France                   8341      209024.05   25.06     16.6   74.36      43.50   296.7
Australia                1182      138521.31  117.19     66.0  159.90       3.43   136.4
Spain                    2484       61577.11   24.79     15.0   70.35      14.15   283.8
Switzerland              1841       56443.95   30.66     17.7   35.82       3.73   116.8
Belgium                  2031       41196.34   20.28     16.6   15.25       3.93    75.2
Sweden               

In [10]:
# Comparación directa entre UK y el segundo país
pais_2 = top_paises[1]

uk   = df[df['Country'] == 'United Kingdom']['Revenue']
otro = df[df['Country'] == pais_2]['Revenue']

print(f"{'Medida':<22} {'United Kingdom':>16} {pais_2:>16}")
print("─" * 56)
for nombre, val_uk, val_otro in [
    ("n transacciones",  f"{len(uk):,}",              f"{len(otro):,}"),
    ("Media (£)",        f"{uk.mean():.2f}",           f"{otro.mean():.2f}"),
    ("Mediana (£)",      f"{uk.median():.2f}",         f"{otro.median():.2f}"),
    ("Desvío (£)",       f"{uk.std():.2f}",            f"{otro.std():.2f}"),
    ("CV (%)",           f"{uk.std()/uk.mean()*100:.1f}%", f"{otro.std()/otro.mean()*100:.1f}%"),
    ("Asimetría G1",     f"{uk.skew():.4f}",           f"{otro.skew():.4f}"),
]:
    print(f"{nombre:<22} {val_uk:>16} {val_otro:>16}")

Medida                   United Kingdom      Netherlands
────────────────────────────────────────────────────────
n transacciones                 354,321            2,359
Media (£)                         20.63           121.00
Mediana (£)                       10.20            91.80
Desvío (£)                       326.04           164.05
CV (%)                          1580.7%           135.6%
Asimetría G1                   431.7927          13.6182


---
## 6. Análisis temporal — Revenue por mes

Usamos estadística descriptiva para detectar **estacionalidad**: si algunos meses tienen distribuciones de revenue distintas.

In [ ]:
meses = {1:'Ene',2:'Feb',3:'Mar',4:'Abr',5:'May',6:'Jun',
         7:'Jul',8:'Ago',9:'Sep',10:'Oct',11:'Nov',12:'Dic'}

resumen_mes = (df.groupby('Mes')['Revenue']
               .agg(
                   transacciones='count',
                   revenue_total='sum',
                   media='mean',
                   mediana='median',
                   desvio='std',
                   asimetria='skew'
               )
               .round(2))

resumen_mes.index = resumen_mes.index.map(meses)
resumen_mes['CV (%)'] = (resumen_mes['desvio'] / resumen_mes['media'] * 100).round(1)

print(resumen_mes.to_string())

mes_max = resumen_mes['revenue_total'].idxmax()
mes_min = resumen_mes['revenue_total'].idxmin()
print(f"\nMes con mayor revenue total: {mes_max}")
print(f"Mes con menor revenue total: {mes_min}")

print("\n💡 ¿Hay un pico de ventas hacia fin de año? (típico en artículos de regalo)")

---
## 7. Análisis de devoluciones

Las devoluciones son una parte importante del negocio. ¿Tienen algún patrón estadístico?

In [11]:
dev = devoluciones.copy()
dev['Quantity'] = dev['Quantity'].abs()         # convertimos a positivo para analizar magnitudes
dev['Revenue']  = dev['Quantity'] * dev['UnitPrice'].abs()

print(f"Total de devoluciones: {len(dev):,} transacciones")
print(f"Revenue devuelto total: £{dev['Revenue'].sum():,.2f}")
print(f"\nPorcentaje de devoluciones sobre ventas:")
print(f"  Por transacciones: {len(dev)/len(df)*100:.1f}%")
print(f"  Por revenue:       {dev['Revenue'].sum()/df['Revenue'].sum()*100:.1f}%")

print("\n── Estadísticas del revenue devuelto ──")
print(f"  Media   : £{dev['Revenue'].mean():.2f}")
print(f"  Mediana : £{dev['Revenue'].median():.2f}")
print(f"  G1      : {dev['Revenue'].skew():.4f}")

print(f"\nTop 5 países con más devoluciones:")
print(dev.groupby('Country')['Revenue']
         .agg(n='count', total='sum')
         .sort_values('total', ascending=False)
         .head(5)
         .round(2)
         .to_string())

Total de devoluciones: 9,288 transacciones
Revenue devuelto total: £896,812.49

Porcentaje de devoluciones sobre ventas:
  Por transacciones: 2.3%
  Por revenue:       10.1%

── Estadísticas del revenue devuelto ──
  Media   : £96.56
  Mediana : £8.50
  G1      : 67.5068

Top 5 países con más devoluciones:
                   n      total
Country                        
United Kingdom  7856  815291.60
EIRE             302   20177.14
France           149   12311.21
Singapore          7   12158.90
Germany          453    7168.93


---
## 8. Correlaciones entre variables numéricas

In [12]:
print("── Matriz de correlación (Pearson) ──")
print(df[['Quantity', 'UnitPrice', 'Revenue']].corr().round(4))

print("\n── Correlación de Spearman (más robusta ante outliers) ──")
print(df[['Quantity', 'UnitPrice', 'Revenue']].corr(method='spearman').round(4))

print("\n💡 Pearson vs Spearman:")
print("   Pearson mide correlación LINEAL — sensible a outliers.")
print("   Spearman mide correlación MONÓTONA — trabaja sobre rangos, más robusta.")
print("   Con distribuciones muy asimétricas como esta, Spearman suele ser más informativo.")

── Matriz de correlación (Pearson) ──
           Quantity  UnitPrice  Revenue
Quantity     1.0000    -0.0046   0.9144
UnitPrice   -0.0046     1.0000   0.0816
Revenue      0.9144     0.0816   1.0000

── Correlación de Spearman (más robusta ante outliers) ──
           Quantity  UnitPrice  Revenue
Quantity     1.0000    -0.4077   0.6573
UnitPrice   -0.4077     1.0000   0.3487
Revenue      0.6573     0.3487   1.0000

💡 Pearson vs Spearman:
   Pearson mide correlación LINEAL — sensible a outliers.
   Spearman mide correlación MONÓTONA — trabaja sobre rangos, más robusta.
   Con distribuciones muy asimétricas como esta, Spearman suele ser más informativo.


---
## 9. Resumen

In [14]:
print("╔" + "═"*54 + "╗")
print("║   RESUMEN  — Online Retail UK              ║")
print("╠" + "═"*54 + "╣")
print(f"║  Período       : {df['InvoiceDate'].min().date()} → {df['InvoiceDate'].max().date()}  ║")
print(f"║  Transacciones : {len(df):>10,}                        ║")
print(f"║  Clientes      : {df['CustomerID'].nunique():>10,}                        ║")
print(f"║  Países        : {df['Country'].nunique():>10,}                        ║")
print(f"║  Revenue total : £{df['Revenue'].sum():>10,.2f}                  ║")
print("╠" + "═"*54 + "╣")
print("║  REVENUE POR TRANSACCIÓN                            ║")
print(f"║  Media         : £{df['Revenue'].mean():>10.2f}   (sensible a outliers) ║")
print(f"║  Mediana       : £{df['Revenue'].median():>10.2f}   (robusta)            ║")
print(f"║  Desvío        : £{df['Revenue'].std():>10.2f}                        ║")
print(f"║  CV            : {df['Revenue'].std()/df['Revenue'].mean()*100:>9.1f}%                         ║")
print(f"║  Asimetría G1  : {df['Revenue'].skew():>10.4f}   (alta asim. positiva) ║")
print("╠" + "═"*54 + "╣")
print("║  CONCLUSIONES CLAVE                                 ║")
print("║  • La media del revenue NO representa al cliente    ║")
print("║    típico → usar MEDIANA para reportes operativos   ║")
print("║  • Alto CV indica enorme variabilidad entre pedidos ║")
print("║  • Distribución de Pareto: ~20% genera ~80% revenue ║")
print("║  • UK domina el revenue; pico en nov/dic (regalos)  ║")
print("╚" + "═"*54 + "╝")

╔══════════════════════════════════════════════════════╗
║   RESUMEN  — Online Retail UK              ║
╠══════════════════════════════════════════════════════╣
║  Período       : 2010-12-01 → 2011-12-09  ║
║  Transacciones :    397,884                        ║
║  Clientes      :      4,338                        ║
║  Países        :         37                        ║
║  Revenue total : £8,911,407.90                  ║
╠══════════════════════════════════════════════════════╣
║  REVENUE POR TRANSACCIÓN                            ║
║  Media         : £     22.40   (sensible a outliers) ║
║  Mediana       : £     11.80   (robusta)            ║
║  Desvío        : £    309.07                        ║
║  CV            :    1380.0%                         ║
║  Asimetría G1  :   451.4432   (alta asim. positiva) ║
╠══════════════════════════════════════════════════════╣
║  CONCLUSIONES CLAVE                                 ║
║  • La media del revenue NO representa al cliente    ║
║    típico →

---
## ✏️ Ejercicios

**Ejercicio 1.** Calculá el revenue **promedio por cliente** (no por transacción). ¿Cómo se compara la media con la mediana? ¿Qué conclusión de negocio sacás?

**Ejercicio 2.** Filtrá solo las transacciones del **Reino Unido** y compará la distribución de `UnitPrice` entre los meses de mayor y menor venta. ¿Hay diferencias en la asimetría?

**Ejercicio 3.** Encontrá los **10 productos más vendidos** (por quantity total) y calculá su precio promedio, mediana y CV. ¿Los productos más vendidos son los más baratos?

**Ejercicio 4.** Calculá el **ticket promedio por día de semana** (revenue total / cantidad de transacciones). ¿Qué día tiene mayor ticket promedio? ¿Coincide con la mediana?

**Ejercicio 5.** Usando la regla de Tukey, detectá outliers en `Revenue` para el mercado UK. ¿Qué porcentaje de las transacciones son outliers? ¿Qué impacto tienen en la media?

In [ ]:
# Ejercicio 1


In [ ]:
# Ejercicio 2


In [ ]:
# Ejercicio 3


In [ ]:
# Ejercicio 4


In [ ]:
# Ejercicio 5
